NOTE: This experiment was a failure! Projector splitting in poorly suited to diffusive processes, and the fact that they decay quickly to lower rank means it's easy to "over-rank" the solution, leading to small singular values and blow-up.

# Dynamical Low-Rank Approximation for the 2D Heat Equation

**Notes:** 
- This implementation using the *projector-splitting integrator(s)* method. Here I use first-order Lie-Trotter splitting.
- Doing this involves solving a sequence of time ODEs in turn. Here I use the forward Euler method: Given $\dot{f}$ and $f(t_n)$, find,

$$
    f(t_{n+1}) = f(t_n) + \dot{f}(t_n)\Delta t
$$

- For the discrete Laplacian, I use the second order central difference

$$
    \partial_x u(x_i) \approx \frac{u_{i-1} - 2u_i + u_{i+1}}{h}
$$

## Problem
$$
\begin{aligned}
    u_t &= -\nabla^2 u \\\\
    u |_{\partial \Omega} &= 0 \\\\
    u(0, x, y) &= \max(1 - |2x - 1|, 0) \cdot \max(1 - |2y - 1|, 0) \\\\
    \Omega &= [0,1] \times [0,1]
\end{aligned}
$$

In [1]:
import numpy as np

In [2]:
# initial condition
# boundary conditions will be enforce implicitly
u0 = lambda x, y: max(1 - np.abs(2 * x - 1), 0) * max(1 - np.abs(2 * y - 1), 0)

In [17]:
# Spatial domain
x_range = (0,1)
y_range = (0,1)

# Number of grid points in each direction
Nx = 5
Ny = 5

dx = (x_range[1] - x_range[0]) / Nx
dy = (y_range[1] - y_range[0]) / Ny

# Resulting grid per axis
x_Nx = np.linspace(x_range[0], x_range[1], Nx)
y_Ny = np.linspace(y_range[0], y_range[1], Ny)

In [18]:
# discretized initial value
u0_N = np.array([[u0(x,y) for y in y_Ny] for x in x_Nx])

In [19]:
# time range
t_range = (0,1)
# number of steps
Nt = 30
# step size
dt = (t_range[1] - t_range[0]) / Nt

In [20]:
# Desired rank of our solution.
rank = 1
# Data arrays for our low-rank approximation
# The first index for each of these points to the time step of our solution
# The result at time step t is calculated as X[t] @ S[t] @ Y[t].T, just as in the usual SVD
X = np.zeros((Nt, Nx, rank))
S = np.zeros((Nt, rank, rank))
Y = np.zeros((Nt, Ny, rank))

In [21]:
# Initialize at time 0 using SVD of discretized initial condition
X0, S0, Y0t = np.linalg.svd(u0_N)
# Truncate to desired rank and reorient layout as needed
X[0] = X0[:, :rank]
S[0] = np.diag(S0[:rank])
Y[0] = Y0t[:rank, :].T

In [22]:
# Discrete Laplacian for x and y directions
# Central difference
D_xx = (1 / dx**2) * (np.diag(np.ones(Nx - 1), -1) + np.diag(np.ones(Nx - 1), 1) + np.diag(np.full(Nx, -2), 0))
D_yy = (1 / dy**2) * (np.diag(np.ones(Ny - 1), -1) + np.diag(np.ones(Ny - 1), 1) + np.diag(np.full(Ny, -2), 0))
# Enforce boundary condition
D_xx[0,0] = 0
D_xx[Nx - 1, Nx - 1] = 0
D_yy[0,0] = 0
D_yy[Ny - 1, Ny - 1] = 0

In [23]:
# Time-stepping
for t in range(Nt - 1):
    # Coefficient matrice for the projection 
    # of Y[t] and X[t] onto the spans of their Laplacian
    M = Y[t].T @ D_yy @ Y[t]
    N = X[t].T @ D_xx @ X[t]
    # K-step
    K = X[t] @ S[t]
    # Approximate time derivative of K
    K_dot = D_xx @ K - K @ M
    # Fwd Euler to estimate K[t+1]
    K = K + K_dot * dt
    # Orthonormal decomposition of K
    X_new, S_new = np.linalg.qr(K)
    # The result for the x-component is the 
    # value for the next time step.
    # The result for S is an intermediate value
    # used in the next step
    X[t + 1] = X_new

    # S-step
    # Discrete approximation of the projection of
    # XY onto the span of XSY^T
    S_dot = -(N @ S_new + S_new @ M)
    # Fwd Euler estimate for update to S
    S_new = S_new + S_dot * dt

    # print(S_new[0,0] / S_new[1,1])

    # L-step
    L = S_new @ Y[t].T
    # Discrete approximation for the projection 
    # of X onto the span of SY^T
    L_dot = -(N @ L) + L @ D_yy
    # Fwd Euler estimate for update to L
    L = L + L_dot * dt
    # Orthonormalization of L
    Y_new, S_new = np.linalg.qr(L.T)

    # print(S_new[0,0] / S_new[1,1])
    # print("")

    # Last update for this iteration
    S[t + 1] = S_new
    Y[t + 1] = Y_new

In [24]:
def u_hat(t):
    return X[t] @ S[t] @ Y[t].T

In [25]:
u_hat(0)

array([[0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.25, 0.5 , 0.25, 0.  ],
       [0.  , 0.5 , 1.  , 0.5 , 0.  ],
       [0.  , 0.25, 0.5 , 0.25, 0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  ]])

In [26]:
u_hat(1)

array([[0.36651235, 0.68415638, 0.63528807, 0.68415638, 0.36651235],
       [0.68415638, 1.27709191, 1.18587106, 1.27709191, 0.68415638],
       [0.63528807, 1.18587106, 1.10116598, 1.18587106, 0.63528807],
       [0.68415638, 1.27709191, 1.18587106, 1.27709191, 0.68415638],
       [0.36651235, 0.68415638, 0.63528807, 0.68415638, 0.36651235]])

In [27]:
u_hat(2)

array([[2.18360245, 0.84222626, 1.64397976, 0.84222626, 2.18360245],
       [0.84222626, 0.32485084, 0.63409112, 0.32485084, 0.84222626],
       [1.64397976, 0.63409112, 1.23771131, 0.63409112, 1.64397976],
       [0.84222626, 0.32485084, 0.63409112, 0.32485084, 0.84222626],
       [2.18360245, 0.84222626, 1.64397976, 0.84222626, 2.18360245]])

In [28]:
u_hat(3)

array([[ 1.01623917,  1.0716158 , -0.07128195,  1.0716158 ,  1.01623917],
       [ 1.0716158 ,  1.13001001, -0.07516623,  1.13001001,  1.0716158 ],
       [-0.07128195, -0.07516623,  0.00499992, -0.07516623, -0.07128195],
       [ 1.0716158 ,  1.13001001, -0.07516623,  1.13001001,  1.0716158 ],
       [ 1.01623917,  1.0716158 , -0.07128195,  1.0716158 ,  1.01623917]])

In [29]:
u_hat(4)

array([[4.84584777, 0.44732777, 4.38811591, 0.44732777, 4.84584777],
       [0.44732777, 0.04129353, 0.40507383, 0.04129353, 0.44732777],
       [4.38811591, 0.40507383, 3.97362076, 0.40507383, 4.38811591],
       [0.44732777, 0.04129353, 0.40507383, 0.04129353, 0.44732777],
       [4.84584777, 0.44732777, 4.38811591, 0.44732777, 4.84584777]])

In [30]:
u_hat(5)

array([[14.1140197 , 16.09058131, -1.98661214, 16.09058131, 14.1140197 ],
       [16.09058131, 18.34394542, -2.26482213, 18.34394542, 16.09058131],
       [-1.98661214, -2.26482213,  0.27962465, -2.26482213, -1.98661214],
       [16.09058131, 18.34394542, -2.26482213, 18.34394542, 16.09058131],
       [14.1140197 , 16.09058131, -1.98661214, 16.09058131, 14.1140197 ]])

As can be seen, the result is quite unstable. According to AI, this is due to the diffusive nature of the heat equation and the unidirectional flow of information -- the S-step in projector-splitting is still susceptible to instability for very small singular values. Even with rank 1, and with $\Delta t << \frac{1}{2(1/(\Delta x)^2 + 1/(\Delta y)^2)}$, we have noise introduced by the "backward-looking" nature of the S integrator step.